In [ ]:
import os
import re
from tqdm import tqdm
from bs4 import BeautifulSoup
import time
import uuid
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import cloudscraper
from urllib.parse import quote_plus
from fake_useragent import UserAgent

# Initialize fake user agent generator
ua = UserAgent()

# Generate headers with a random User-Agent
headers = {
    "User-Agent": ua.random,
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/",
}


scraper = cloudscraper.create_scraper()





def clean_title(title):
    """
    Remove parentheses and everything inside from the title.
    Example: 'The Diary of a Young Girl (Mass Market Paperback)' -> 'The Diary of a Young Girl'
    """
    return re.sub(r"\s*\(.*?\)", "", title).strip()

    
def downld_epub_fast(epub_link, scraper, download_dir="download_dir", chunk_size=65536, max_workers=4):
    """
    Fast EPUB downloader with multiple optimizations:
    - Larger chunk size (64KB default)
    - Parallel chunk downloading for large files
    - Reduced system calls
    - Optimized file I/O
    """
    try:
        os.makedirs(download_dir, exist_ok=True)
        
        # First, get file info with HEAD request (faster than GET for metadata)
        head_response = scraper.head(epub_link, timeout=30)
        head_response.raise_for_status()
        
        # Extract filename from Content-Disposition
        cd = head_response.headers.get("content-disposition", "")
        match = re.search(r'filename="?([^"]+)"?', cd)
        if match:
            raw_name = match.group(1)
            final_filename = os.path.basename(raw_name.strip('"'))
        else:
            print("❌ No valid filename in headers")
            return None
            
        save_path = os.path.join(download_dir, final_filename)
        
        # Skip if already exists
        if os.path.exists(save_path):
            print(f"⏭️  File already exists: {save_path}")
            return save_path
            
        total_size = int(head_response.headers.get("content-length", 0))
        
        # For small files or when parallel download isn't beneficial, use simple download
        if total_size < 10 * 1024 * 1024:  # Less than 10MB
            return _simple_fast_download(epub_link, scraper, save_path, final_filename, total_size, chunk_size)
        
            
    except Exception as e:
        print(f"❌ Download failed: {e}")
        return None

def _simple_fast_download(epub_link, scraper, save_path, filename, total_size, chunk_size):
    """Optimized single-threaded download for smaller files"""
    with scraper.get(epub_link, stream=True, timeout=30) as response:
        response.raise_for_status()
        
        with open(save_path, "wb") as file, tqdm(
            desc=filename,
            total=total_size,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
        ) as bar:
            # Write chunks in larger batches to reduce system calls
            buffer = bytearray()
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    buffer.extend(chunk)
                    # Write buffer when it gets large enough
                    if len(buffer) >= chunk_size * 4:  # Write every ~256KB
                        file.write(buffer)
                        bar.update(len(buffer))
                        buffer.clear()
            
            # Write remaining buffer
            if buffer:
                file.write(buffer)
                bar.update(len(buffer))
    
    print(f"✅ Download complete: {save_path}")
    return save_path


def fetch_and_download(payload, scraper=None, download_dir="download_dir", max_retries=3):
    """
    Given a payload {id, filename}, handle the whole process:
      - POST to Fetching_Resource.php
      - Extract redirect link
      - Validate headers
      - Download if valid
    Returns the saved file path or None.
    Retries HEAD request failures with exponential backoff (5s, 10s, 20s).
    """
    import re
    import time

    base_url = "https://oceanofpdf.com/Fetching_Resource.php"

    # Use provided scraper or create a new one
    scraper = scraper or cloudscraper.create_scraper()

    print(f"\n[+] Requesting resource for {payload['filename']}...")
    try:
        # Step 1: submit the form
        response = scraper.post(base_url, data=payload, timeout=20)
        response.raise_for_status()
    except Exception as e:
        print(f"❌ POST request failed: {e}")
        return None

    # Step 2: look for redirect link
    match = re.search(r'https://fs\d+\.oceanofpdf\.com/[^\s"\']+', response.text)
    if not match:
        print("[!] No redirect URL found. Response preview:")
        print(response.text[:500])
        return None

    redirect_url = match.group(0)

    # Step 3: HEAD request with retries
    attempt = 0
    head_resp = None
    while attempt < max_retries:
        try:
            head_resp = scraper.head(redirect_url, allow_redirects=True, timeout=15)
            head_resp.raise_for_status()
            break  # success, exit loop
        except Exception as e:
            attempt += 1
            if attempt < max_retries:
                wait_time = 5 * (2 ** (attempt - 1))  # 5s, 10s, 20s backoff
                print(f"❌ HEAD request failed (attempt {attempt}/{max_retries}): {e}")
                print(f"   Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                print(f"❌ HEAD request failed after {max_retries} attempts: {e}")
                return None

    # Step 4: validate content-disposition
    cd = head_resp.headers.get("content-disposition", "")
    if "attachment" in cd and "filename=" in cd:
        return downld_epub_fast(
            redirect_url, scraper,
            download_dir=download_dir,
            chunk_size=65536,
            max_workers=4
        )
    else:
        print("❌ No valid downloadable attachment in headers.")
        return None



def get_download_forms(book_url, scraper):
    """
    Fetch all download form details (id, filename) from a book page.
    Returns only EPUB forms if available, otherwise returns other formats.
    Returns a list of dicts like:
        [{"id": "srv3", "filename": "Book.epub"}, {"id": "srv4", "filename": "Book.pdf"}]
    """
    try:
        response = scraper.get(book_url,headers=headers, timeout=15)
        response.raise_for_status()
    except Exception as e:
        print(f"[!] Failed to fetch {book_url}: {e}")
        return []
    
    soup = BeautifulSoup(response.text, "html.parser")
    forms = soup.find_all("form", action="https://oceanofpdf.com/Fetching_Resource.php")
    
    epub_forms = []
    other_forms = []
    
    for form in forms:
        id_input = form.find("input", {"name": "id"})
        filename_input = form.find("input", {"name": "filename"})
        
        if id_input and filename_input:
            file_ext = filename_input["value"].split(".")[-1].lower()
            form_data = {
                "id": id_input["value"],
                "filename": filename_input["value"]
            }
            
            if file_ext == "epub":
                epub_forms.append(form_data)
            else:
                other_forms.append(form_data)
    
    # Return only EPUB forms if available, otherwise return other formats
    if epub_forms:
        time.sleep(3)  # throttle requests
        return epub_forms
    else:
        time.sleep(3)  # throttle requests
        return other_forms


def get_last_page(url):
    """Find the last page number from pagination."""
    try:
        print(f"getting last page for {url}")
        response = scraper.get(url,headers=headers, timeout=15)
        response.raise_for_status()
    except Exception as e:
        print(f"[!] Failed to fetch {url}: {e}")
        return 1  # fallback: only page 1

    soup = BeautifulSoup(response.text, "html.parser")
    pagination_div = soup.find("div", class_="archive-pagination pagination")

    if not pagination_div:
        return 1

    page_numbers = []
    for a_tag in pagination_div.find_all("a", href=True):
        # remove <span> tags
        for span in a_tag.find_all("span"):
            span.decompose()

        text = a_tag.get_text(strip=True)
        if text.isdigit():
            page_numbers.append(int(text))

    time.sleep(3)

    return max(page_numbers) if page_numbers else 1




def Download_books_from_Oceanofpdf_Category(base_url, start_page=None, stop_page=None, max_pages=None, First_N_books=None):
    """
    Fetch all book article links across pagination pages.
    Supports optional start_page, stop_page, max_pages, and First_N_books limits.
    Keeps record of failed downloads in a text file.
    """
    
    def log_failed_book(entry):
        """Immediately append a failed book entry to log file."""
        os.makedirs("logs", exist_ok=True)
        fail_log = os.path.join("logs", "failed_books_11.txt")
        mode = "a" if os.path.exists(fail_log) else "w"
        with open(fail_log, mode, encoding="utf-8") as f:
            if mode == "w":  # first time create
                f.write(f"--- New session: {time.strftime('%Y-%m-%d %H:%M:%S')} ---\n")
            f.write(str(entry) + "\n")
        print(f"📄 Logged failed book immediately: {entry}")
        
    category_name = None
    
    if base_url.startswith('https://oceanofpdf.com/category/genres/'):
        category_name = base_url.replace('https://oceanofpdf.com/category/genres/','').strip()

    # Detect last page if stop_page or max_pages not provided
    last_page = get_last_page(base_url)
    print(f"Detected last page: {last_page}")

    # Set defaults
    start_page = start_page or 1
    stop_page = stop_page or last_page

    # Apply max_pages if supplied
    if max_pages:
        stop_page = min(start_page + max_pages - 1, stop_page)

    print(f"Fetching from page {start_page} to {stop_page}")

    downloaded_count = 0
    all_links = []
    failed_books = []  # keep record of failed downloads

    for page in range(start_page, stop_page + 1):
        page_url = f"{base_url}page/{page}/"
        print(f"[+] Fetching page {page}: {page_url}")

        try:
            response = scraper.get(page_url, timeout=15)
            response.raise_for_status()
        except Exception as e:
            print(f"[!] Failed to fetch {page_url}: {e}")
            continue

        soup = BeautifulSoup(response.text, "html.parser")

        # If category_name not set, fetch from page
        if category_name is None:
            h1_tag = soup.select_one(
                "div.archive-description.taxonomy-archive-description.taxonomy-description > h1.archive-title"
            )
            if h1_tag:
                category_name = h1_tag.text.strip().replace(" ", "-")
                print(f"[+] Detected category name: {category_name}")

        articles = soup.find_all("article")

        for article in articles:
            if First_N_books and downloaded_count >= First_N_books:
                print(f"\n✅ Reached limit of {First_N_books} books. Stopping.")
                # Save failed books before returning
                if failed_books:
                    with open(f"failed_books_{category_name}.txt", "w", encoding="utf-8") as f:
                        for fail in failed_books:
                            f.write(fail + "\n")
                return all_links

            postmetainfo = article.find("div", class_="postmetainfo")
            if postmetainfo:
                language_strong = postmetainfo.find("strong", string="Language: ")
                if language_strong:
                    language_text = language_strong.next_sibling
                    if language_text.strip().lower() != "english":
                        print(f"❌ Skipping non-English book")
                        continue  # Skip non-English books

            header = article.find("header", class_="entry-header")
            if header:
                a_tag = header.find("a", class_="entry-title-link", href=True)
                if a_tag:
                    book_url = a_tag["href"]
                    all_links.append(book_url)
                    payload_list = get_download_forms(book_url, scraper)
                    if not payload_list:
                        print("❌ No forms found on page.")
                        failed_books.append(book_url)
                        continue

                    if isinstance(payload_list, dict):
                        path = fetch_and_download(payload_list, scraper, download_dir=f"download_dir_{category_name}")
                        if path:
                            downloaded_count += 1
                            print(f"📥 Downloaded books: {downloaded_count}")
                        else:
                            failed_books.append(book_url)
                    elif isinstance(payload_list, list):
                        success = False
                        for payload in payload_list:
                            path = fetch_and_download(payload, scraper, download_dir=f"download_dir_{category_name}")
                            if path:
                                downloaded_count += 1
                                success = True
                                print(f"📥 Downloaded books: {downloaded_count}")
                                break  # stop after first successful format
                        if not success:
                            failed_books.append(book_url)

        time.sleep(5)

    # Save failed books after all pages processed
    if failed_books:
        with open(f"failed_books_{category_name}.txt", "w", encoding="utf-8") as f:
            for fail in failed_books:
                f.write(fail + "\n")
        print(f"\n⚠️ Failed books saved to failed_books_{category_name}.txt")

    print(f"\nTotal Book Links Collected: {len(all_links)}")
    return all_links


In [ ]:
if __name__ == "__main__":
    url = "https://oceanofpdf.com/category/genres/harlequin-romance/"
    
    print(f"\nFetching books from: {url}")
    book_links = Download_books_from_Oceanofpdf_Category(base_url=url)
    
    print("\n=== All Book Links Collected ===")



In [ ]:
import os

path = r"C:\cracks\My_App\epub-library-manager\upload"

result = []

for name in os.listdir(path):
    folder_path = os.path.join(path, name)
    if os.path.isdir(folder_path):
        # Count only .pdf and .epub files inside the folder (not subfolders)
        count = sum(
            1 for f in os.listdir(folder_path)
            if os.path.isfile(os.path.join(folder_path, f)) and f.lower().endswith((".pdf", ".epub"))
        )
        
        if count < 10:
            result.append(name.replace("_", " "))

print(result)

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
from datetime import datetime

def scrape_goodreads_categories_by_year(year):
    """
    Scrape Goodreads Choice Awards categories for a specific year
    """
    url = f"https://www.goodreads.com/choiceawards/best-books-{year}"
    
    # Add headers to mimic a browser request
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    try:
        print(f"Scraping {year}... ", end="", flush=True)
        
        # Send GET request
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        
        # Parse HTML content
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Find all category containers
        category_containers = soup.find_all('div', class_='category clearFix')
        
        categories = []
        
        for container in category_containers:
            # Find the category link and name
            category_link = container.find('a', href=True)
            
            if category_link:
                # Extract category name from the h4 tag
                category_name = category_link.find('h4', class_='category__copy')
                
                if category_name:
                    name = category_name.get_text(strip=True)
                    # Convert relative URL to absolute URL
                    link = "https://www.goodreads.com" + category_link['href']
                    
                    categories.append({
                        'Year': year,
                        'Category': name,
                        'Link': link
                    })
        
        print(f"Found {len(categories)} categories")
        return categories
    
    except requests.exceptions.RequestException as e:
        print(f"Error fetching page for {year}: {e}")
        return []
    except Exception as e:
        print(f"Error parsing page for {year}: {e}")
        return []

def scrape_all_years(start_year=2011, end_year=2024):
    """
    Scrape categories for all years from start_year to end_year
    """
    all_categories = []
    
    print("Starting multi-year scraping...")
    print("=" * 60)
    
    for year in range(start_year, end_year + 1):
        categories = scrape_goodreads_categories_by_year(year)
        all_categories.extend(categories)
        
        # Add a small delay to be respectful to the server
        time.sleep(1)
    
    return all_categories

def analyze_categories_by_year(df):
    """
    Analyze and display category trends across years
    """
    print("\nCategory Analysis:")
    print("=" * 40)
    
    # Categories per year
    categories_per_year = df.groupby('Year')['Category'].count().reset_index()
    categories_per_year.columns = ['Year', 'Number_of_Categories']
    
    print("\nNumber of categories per year:")
    print(categories_per_year.to_string(index=False))
    
    # Most common categories across all years
    print("\nMost common categories across all years:")
    category_counts = df['Category'].value_counts().head(10)
    print(category_counts.to_string())
    
    # Categories that appeared in specific years
    print("\nCategories by year they appeared:")
    category_years = df.groupby('Category')['Year'].apply(list).reset_index()
    category_years['Years_Active'] = category_years['Year'].apply(lambda x: f"{min(x)}-{max(x)}" if len(x) > 1 else str(x[0]))
    category_years['Total_Years'] = category_years['Year'].apply(len)
    category_years = category_years.sort_values('Total_Years', ascending=False)
    
    print(category_years[['Category', 'Years_Active', 'Total_Years']].to_string(index=False))

def save_results(all_categories):
    """
    Save results to multiple formats
    """
    if not all_categories:
        print("No categories found to save.")
        return None
    
    # Create DataFrame
    df = pd.DataFrame(all_categories)
    
    # Create timestamp for filenames
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Save complete data
    complete_filename = f'goodreads_categories_2011-2024_{timestamp}.csv'
    df.to_csv(complete_filename, index=False)
    print(f"\nComplete data saved to: {complete_filename}")
    
    # Save year-by-year summary
    summary_filename = f'goodreads_categories_summary_{timestamp}.csv'
    year_summary = df.groupby(['Year', 'Category']).size().reset_index(name='Count')
    year_summary.to_csv(summary_filename, index=False)
    print(f"Year summary saved to: {summary_filename}")
    
    # Create separate files for each year
    years_dir = f'goodreads_by_year_{timestamp}'
    os.makedirs(years_dir, exist_ok=True)
    
    for year in df['Year'].unique():
        year_data = df[df['Year'] == year][['Category', 'Link']]
        year_filename = os.path.join(years_dir, f'goodreads_{year}.csv')
        year_data.to_csv(year_filename, index=False)
    
    print(f"Individual year files saved in: {years_dir}/")
    
    return df

def main():
    print("Goodreads Choice Awards Multi-Year Scraper (2011-2024)")
    print("=" * 60)
    
    # Scrape all years
    all_categories = scrape_all_years(2011, 2024)
    
    if all_categories:
        print(f"\nTotal categories found across all years: {len(all_categories)}")
        
        # Save results
        df = save_results(all_categories)
        
        if df is not None:
            # Analyze trends
            analyze_categories_by_year(df)
            
            print(f"\nScraping completed successfully!")
            print(f"Data spans from 2011 to 2024")
            print(f"Total records: {len(df)}")
            print(f"Unique categories: {df['Category'].nunique()}")
    else:
        print("No categories were found across any years.")

if __name__ == "__main__":
    main()

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import os

BASE_URL = "https://www.goodreads.com"
START_URL = "https://www.goodreads.com/list/show/153860.Goodreads_Top_100_Highest_Rated_Books_on_Goodreads_with_at_least_10_000_Ratings"
CSV_FILE = "Goodreads_Top_100_Highest_Rated_Books_on_Goodreads_with_at_least_10_000_Ratings.csv"

def scrape_goodreads_list(url, max_pages=100):
    page = 1
    file_exists = os.path.isfile(CSV_FILE)

    # if file exists, delete it so we start fresh
    if file_exists:
        os.remove(CSV_FILE)

    while url and page <= max_pages:
        print(f"Scraping page {page}: {url}")
        res = requests.get(url, headers={
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
        })
        if res.status_code != 200:
            print(f"❌ Failed to fetch {url}, status: {res.status_code}")
            break

        soup = BeautifulSoup(res.text, "html.parser")

        # find the main book table
        table = soup.find("table", {"class": "tableList js-dataTooltip"})
        if not table:
            print("⚠️ Could not find the book table")
            break

        rows = table.find_all("tr", {"itemtype": "http://schema.org/Book"})
        page_results = []
        for row in rows:
            book_tag = row.find("a", {"class": "bookTitle"})
            title = book_tag.get_text(strip=True) if book_tag else None
            book_link = BASE_URL + book_tag["href"] if book_tag else None

            author_tag = row.find("a", {"class": "authorName"})
            author = author_tag.get_text(strip=True) if author_tag else None

            if title and author and book_link:
                page_results.append([title, author, book_link])

        # write results to CSV
        write_mode = "w" if page == 1 else "a"
        with open(CSV_FILE, write_mode, newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            if page == 1:  # add header only once
                writer.writerow(["Title", "Author", "Link"])
            writer.writerows(page_results)

        print(f"✅ Saved {len(page_results)} books from page {page}")

        # find next page link
        pagination = soup.find("div", {"class": "pagination"})
        next_page = None
        if pagination:
            next_tag = pagination.find("a", {"class": "next_page"})
            if next_tag and "href" in next_tag.attrs:
                next_page = BASE_URL + next_tag["href"]

        url = next_page
        page += 1

    print(f"🎉 Done! Results saved in {CSV_FILE}")


if __name__ == "__main__":
    scrape_goodreads_list(START_URL, max_pages=2)


In [ ]:
import mysql.connector

# ==== MySQL Connection (XAMPP) ====
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database="final_klaus_ebooks_library"
)
cursor = conn.cursor()

# --- Disable categories with less than 5 books ---
update_category_sql = """
UPDATE categories c
SET c.status = 0
WHERE (
    SELECT COUNT(*) 
    FROM books b 
    WHERE b.cat_id = c.id AND b.status = 1
) < 5
"""
cursor.execute(update_category_sql)
print(f"Categories affected: {cursor.rowcount}")

# --- Disable sub-categories with less than 5 books ---
update_sub_category_sql = """
UPDATE sub_categories sc
SET sc.status = 0
WHERE (
    SELECT COUNT(*) 
    FROM books b 
    WHERE b.sub_cat_id = sc.id AND b.status = 1
) < 5
"""
cursor.execute(update_sub_category_sql)
print(f"Sub-categories affected: {cursor.rowcount}")

# Commit changes
conn.commit()

# Close connection
cursor.close()
conn.close()
